In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: source_0
position:
  x: 0
  y: 77.5
description:
  text: Load CSV file with specified options.
  hash: 4d30ac72
previewCodeHash: 2a17d432d7b0e026
codegenHash: f9bd80fc93b51b62
config:
  file_source:
    path: /Volumes/labuser_srivastavanu2022/get_started_de/sampple_data/turbine_events.csv
    format: '"csv"'
    schemaEvolutionMode: '"none"'
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "file_source": {
        "path": "/Volumes/labuser_srivastavanu2022/get_started_de/sampple_data/turbine_events.csv",
        "format": "\"csv\"",
        "schemaEvolutionMode": "\"none\""
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_0.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_0" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_0.data"])
if "source_0" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_0", "counts": {"data": int(ctx["source_0.data"].count())}}}, raw=True)

In [0]:
"""
id: visualization_4_1
template: visualization
templateVersion: 1.0.0
name: unresolved_incident
position:
  x: 1200
  y: 155
description:
  text: Group data by event type and sum incidents for each type.
  hash: 5c0c10ec
previewCodeHash: 6aa068ec6bbc8ee7
codegenHash: 95fa3a0d85544146
config:
  editorSpec:
    queries:
      - queryName: main_query
        query:
          datasetName: lakeflow_designer_upstream
          fields:
            - expression: "`event_type`"
              fieldName: event_type
            - expression: SUM(`Incident`)
              fieldName: sum(Incident)
          disaggregatedData: false
    renderSpec:
      version: 3
      widgetType: bar
      encodings:
        x:
          fieldName: event_type
          scale:
            type: categorical
        y:
          fieldName: sum(Incident)
          scale:
            type: quantitative
      data:
        queryName: main_query
    parameterQueries: []
    version: 2
  sql_query: |-
    WITH q AS (SELECT * FROM lakeflow_designer_upstream)
    SELECT `event_type` AS event_type, SUM(`Incident`) AS `sum(Incident)`
    FROM q
    GROUP BY event_type
input:
  - node: filter_4
    input_port: data
    output_port: excluded_data
"""

# generated from the system
from typing import Any, Dict

_SOURCE_VIEW = "lakeflow_designer_upstream"

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        return {"data": None}

    sql_query = config.get("sql_query")
    if not isinstance(sql_query, str) or not sql_query.strip():
        return {"data": df}

    try:
        df.createOrReplaceTempView(_SOURCE_VIEW)
        globals().setdefault("_lb_views", set()).add(_SOURCE_VIEW)
        return {"data": spark.sql(sql_query)}
    except Exception as exc:
        print("[visualization] compiled query failed, passing input through:", exc, sql_query)
        return {"data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "editorSpec": {
        "queries": [
            {
                "queryName": "main_query",
                "query": {
                    "datasetName": "lakeflow_designer_upstream",
                    "fields": [
                        {
                            "expression": "`event_type`",
                            "fieldName": "event_type"
                        },
                        {
                            "expression": "SUM(`Incident`)",
                            "fieldName": "sum(Incident)"
                        }
                    ],
                    "disaggregatedData": False
                }
            }
        ],
        "renderSpec": {
            "version": 3,
            "widgetType": "bar",
            "encodings": {
                "x": {
                    "fieldName": "event_type",
                    "scale": {
                        "type": "categorical"
                    }
                },
                "y": {
                    "fieldName": "sum(Incident)",
                    "scale": {
                        "type": "quantitative"
                    }
                }
            },
            "data": {
                "queryName": "main_query"
            }
        },
        "parameterQueries": [],
        "version": 2
    },
    "sql_query": "WITH q AS (SELECT * FROM lakeflow_designer_upstream)\nSELECT `event_type` AS event_type, SUM(`Incident`) AS `sum(Incident)`\nFROM q\nGROUP BY event_type"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_4.excluded_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["visualization_4_1.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "visualization_4_1" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["visualization_4_1.data"])
if "visualization_4_1" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "visualization_4_1", "counts": {"data": int(ctx["visualization_4_1.data"].count())}}}, raw=True)

In [0]:
"""
id: prepare_1
template: prepare
templateVersion: 2.0.0
name: prepare_data
position:
  x: 300
  y: 77.5
description:
  text: Convert event_timestamp to date format and trim spaces from event_type.
  hash: "86123004"
previewCodeHash: e743330fa660314c
codegenHash: d318165b9d528af4
config:
  actions:
    - type: parse_date
      column: event_timestamp
      kind: date
      format: MM-dd-yyyy
      on_error: "null"
    - type: trim
      column: event_type
      side: both
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
import pyspark.sql.functions as F
import pyspark.sql.types as T

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any]):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = F.expr(rendered)
    return df.withColumns(assignments)

def _apply_cast(df, action: Dict[str, Any], column: str):
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {to_type})")
    return _col(column).cast(to_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action_type == "formula":
            df = _apply_formula(df, action)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "parse_date",
            "column": "event_timestamp",
            "kind": "date",
            "format": "MM-dd-yyyy",
            "on_error": "null"
        },
        {
            "type": "trim",
            "column": "event_type",
            "side": "both"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_0.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_1.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_1" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_1.prepared_data"])
if "prepare_1" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_1", "counts": {"prepared_data": int(ctx["prepare_1.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: aggregate_2
template: aggregate
templateVersion: 2.0.0
name: aggregate_2
position:
  x: 600
  y: 77.5
description:
  text: Group by event type, timestamp, and resolved status; count distinct severity as Incident.
  hash: adf2b52b
previewCodeHash: e1d84feb20dffba5
codegenHash: 7e9149ee54d999b7
config:
  group_bys:
    - expr: event_type
      type: column
    - expr: event_timestamp
      type: column
    - expr: resolved
      type: column
  aggregations:
    - columnExpr:
        expr: severity
        type: column
      fn: COUNT_DISTINCT
      alias: Incident
input:
  - node: prepare_1
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        elif fn == "COUNT_DISTINCT":
            arg = F.expr(raw_expr) if col_expr.get("type") != "column" else raw_expr
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "event_type",
            "type": "column"
        },
        {
            "expr": "event_timestamp",
            "type": "column"
        },
        {
            "expr": "resolved",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "severity",
                "type": "column"
            },
            "fn": "COUNT_DISTINCT",
            "alias": "Incident",
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_1.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["aggregate_2.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False) or "aggregate_2" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["aggregate_2.aggregated_data"])
if "aggregate_2" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "aggregate_2", "counts": {"aggregated_data": int(ctx["aggregate_2.aggregated_data"].count())}}}, raw=True)

In [0]:
"""
id: filter_4
template: filter
templateVersion: 2.0.0
name: resolution_state
position:
  x: 900
  y: 77.5
description:
  text: Keep rows where 'resolved' is true; separate others.
  hash: 1203060e
previewCodeHash: 0b751bfd445e5762
codegenHash: d8c91e247e7ed9ff
config:
  condition: resolved IS TRUE
input:
  - node: aggregate_2
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "resolved IS TRUE"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["aggregate_2.aggregated_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_4.filtered_data"] = out["filtered_data"]
ctx["filter_4.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_4" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_4.filtered_data"])
    display(ctx["filter_4.excluded_data"])
if "filter_4" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "filter_4", "counts": {"filtered_data": int(ctx["filter_4.filtered_data"].count()), "excluded_data": int(ctx["filter_4.excluded_data"].count())}}}, raw=True)

In [0]:
"""
id: visualization_4
template: visualization
templateVersion: 1.0.0
name: resolved_incident
position:
  x: 1200
  y: 0
description:
  text: Group data by event type and sum the Incident values.
  hash: 98874d98
previewCodeHash: b98ad04ba5480c1f
codegenHash: 95fa3a0d85544146
config:
  editorSpec:
    queries:
      - queryName: main_query
        query:
          datasetName: lakeflow_designer_upstream
          fields:
            - expression: "`event_type`"
              fieldName: event_type
            - expression: SUM(`Incident`)
              fieldName: sum(Incident)
          disaggregatedData: false
    renderSpec:
      version: 3
      widgetType: bar
      encodings:
        x:
          fieldName: event_type
          scale:
            type: categorical
        y:
          fieldName: sum(Incident)
          scale:
            type: quantitative
      data:
        queryName: main_query
    parameterQueries: []
    version: 2
  sql_query: |-
    WITH q AS (SELECT * FROM lakeflow_designer_upstream)
    SELECT `event_type` AS event_type, SUM(`Incident`) AS `sum(Incident)`
    FROM q
    GROUP BY event_type
input:
  - node: filter_4
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from typing import Any, Dict

_SOURCE_VIEW = "lakeflow_designer_upstream"

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        return {"data": None}

    sql_query = config.get("sql_query")
    if not isinstance(sql_query, str) or not sql_query.strip():
        return {"data": df}

    try:
        df.createOrReplaceTempView(_SOURCE_VIEW)
        globals().setdefault("_lb_views", set()).add(_SOURCE_VIEW)
        return {"data": spark.sql(sql_query)}
    except Exception as exc:
        print("[visualization] compiled query failed, passing input through:", exc, sql_query)
        return {"data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "editorSpec": {
        "queries": [
            {
                "queryName": "main_query",
                "query": {
                    "datasetName": "lakeflow_designer_upstream",
                    "fields": [
                        {
                            "expression": "`event_type`",
                            "fieldName": "event_type"
                        },
                        {
                            "expression": "SUM(`Incident`)",
                            "fieldName": "sum(Incident)"
                        }
                    ],
                    "disaggregatedData": False
                }
            }
        ],
        "renderSpec": {
            "version": 3,
            "widgetType": "bar",
            "encodings": {
                "x": {
                    "fieldName": "event_type",
                    "scale": {
                        "type": "categorical"
                    }
                },
                "y": {
                    "fieldName": "sum(Incident)",
                    "scale": {
                        "type": "quantitative"
                    }
                }
            },
            "data": {
                "queryName": "main_query"
            }
        },
        "parameterQueries": [],
        "version": 2
    },
    "sql_query": "WITH q AS (SELECT * FROM lakeflow_designer_upstream)\nSELECT `event_type` AS event_type, SUM(`Incident`) AS `sum(Incident)`\nFROM q\nGROUP BY event_type"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_4.filtered_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["visualization_4.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "visualization_4" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["visualization_4.data"])
if "visualization_4" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "visualization_4", "counts": {"data": int(ctx["visualization_4.data"].count())}}}, raw=True)